# Advanced House Price Prediction — Targeting MAE ≤ 0.25 Crores

**Current baseline:** MAE ≈ 0.47 (Extra Trees, OHE strategy)

**Techniques applied:**
1. Better feature engineering (price_per_sqft ratio, log area, interaction terms)
2. LightGBM + CatBoost (often outperform XGBoost on tabular data)
3. Deeper XGBoost hyperparameter tuning with RandomizedSearchCV
4. Optuna-based Bayesian hyperparameter optimization
5. Stacking ensemble (XGBoost + LightGBM + RF → Ridge meta-learner)
6. Log-transformed target + back-transform evaluation
7. Full metrics: MAE, RMSE, R², MAPE

In [ ]:
# Install required packages
import subprocess
pkgs = ['xgboost', 'lightgbm', 'catboost', 'optuna', 'category_encoders']
for pkg in pkgs:
    subprocess.run(['pip', 'install', pkg, '-q'])
print('All packages installed.')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import pickle

from sklearn.model_selection import KFold, cross_val_score, train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, StackingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import category_encoders as ce

from xgboost import XGBRegressor
import lightgbm as lgb
from catboost import CatBoostRegressor

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

print('Libraries loaded.')

## 1. Load & Prepare Data

In [ ]:
df = pd.read_csv('gurgaon_properties_post_feature_selection_v2.csv')

# Map furnishing codes
df['furnishing_type'] = df['furnishing_type'].replace({
    0.0: 'unfurnished', 1.0: 'semifurnished', 2.0: 'furnished'
})

print(f'Dataset: {df.shape[0]} rows, {df.shape[1]} cols')
print(f'Price stats (Crores):\n{df["price"].describe()}')

## 2. Feature Engineering

Key insight: `built_up_area` is the dominant feature (~65% importance). Creating derived features from it helps the model learn non-linear price-area relationships.

In [ ]:
def engineer_features(df):
    df = df.copy()
    
    # Log of area (compress large-value outliers)
    df['log_area'] = np.log1p(df['built_up_area'])
    
    # Room density: rooms per unit area
    df['rooms_per_area'] = (df['bedRoom'] + df['bathroom']) / (df['built_up_area'] + 1)
    
    # Total rooms
    df['total_rooms'] = df['bedRoom'] + df['bathroom']
    
    # Has extra amenities
    df['has_servant'] = df['servant room'].clip(0, 1)
    df['has_store'] = df['store room'].clip(0, 1)
    
    # Area bins (captures non-linear price jumps at size boundaries)
    df['area_bin'] = pd.cut(df['built_up_area'],
                            bins=[0, 500, 800, 1200, 1800, 3000, np.inf],
                            labels=['tiny', 'small', 'medium', 'large', 'xlarge', 'luxury'])
    
    return df

df_eng = engineer_features(df)
print('Engineered features added:')
print([c for c in df_eng.columns if c not in df.columns])

In [ ]:
X = df_eng.drop(columns=['price'])
y = df_eng['price']
y_log = np.log1p(y)

# Feature groups
NUM_COLS = ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room',
            'log_area', 'rooms_per_area', 'total_rooms', 'has_servant', 'has_store']
ORD_COLS = ['property_type', 'balcony', 'luxury_category', 'floor_category']
OHE_COLS = ['agePossession', 'furnishing_type', 'area_bin']
TARGET_ENC_COLS = ['sector']

KFOLD = KFold(n_splits=10, shuffle=True, random_state=42)

def get_preprocessor():
    return ColumnTransformer(transformers=[
        ('num',    StandardScaler(),                                NUM_COLS),
        ('ord',    OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), ORD_COLS),
        ('ohe',    OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), OHE_COLS),
        ('target', ce.TargetEncoder(),                              TARGET_ENC_COLS),
    ], remainder='drop')

print(f'X shape: {X.shape}, y range: {y.min():.2f} - {y.max():.2f}')

## 3. Evaluation Helper

In [ ]:
def evaluate_pipeline(name, pipeline, X=X, y_log=y_log):
    """Full evaluation: CV R2, Test MAE, RMSE, R2."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_log, test_size=0.2, random_state=42
    )
    pipeline.fit(X_train, y_train)
    
    y_pred = np.expm1(pipeline.predict(X_test))
    y_true = np.expm1(y_test)
    
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    
    # CV score (on log scale)
    cv_r2 = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='r2', n_jobs=-1).mean()
    
    print(f'{name:35s}  MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}  MAPE={mape:.1f}%  CV_R²={cv_r2:.4f}')
    return {'name': name, 'mae': mae, 'rmse': rmse, 'r2': r2, 'mape': mape, 'cv_r2': cv_r2, 'pipeline': pipeline}

results = []

## 4. Baseline Models (with feature engineering)

In [ ]:
print('=== Baseline with Feature Engineering ===')
print(f'  (Previous best without FE: MAE=0.47)')
print()

for name, model in [
    ('Extra Trees (baseline)', ExtraTreesRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
    ('Random Forest (baseline)', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
    ('XGBoost (baseline)', XGBRegressor(n_estimators=300, random_state=42, n_jobs=-1, verbosity=0)),
    ('LightGBM (baseline)', lgb.LGBMRegressor(n_estimators=300, random_state=42, n_jobs=-1, verbose=-1)),
    ('CatBoost (baseline)', CatBoostRegressor(iterations=300, random_state=42, verbose=0)),
]:
    pipe = Pipeline([('pre', get_preprocessor()), ('reg', model)])
    res = evaluate_pipeline(name, pipe)
    results.append(res)

## 5. XGBoost — Randomized Hyperparameter Search

In [ ]:
from scipy.stats import randint, uniform

xgb_param_dist = {
    'reg__n_estimators'    : randint(200, 800),
    'reg__max_depth'       : randint(3, 10),
    'reg__learning_rate'   : uniform(0.01, 0.19),
    'reg__subsample'       : uniform(0.6, 0.4),
    'reg__colsample_bytree': uniform(0.6, 0.4),
    'reg__min_child_weight': randint(1, 10),
    'reg__gamma'           : uniform(0, 0.3),
    'reg__reg_alpha'       : uniform(0, 0.5),
    'reg__reg_lambda'      : uniform(0.5, 2.0),
}

xgb_pipe = Pipeline([
    ('pre', get_preprocessor()),
    ('reg', XGBRegressor(random_state=42, n_jobs=-1, verbosity=0))
])

X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42)

xgb_search = RandomizedSearchCV(
    xgb_pipe, xgb_param_dist,
    n_iter=50, cv=5, scoring='r2', n_jobs=-1,
    random_state=42, verbose=1
)
xgb_search.fit(X_train, y_train)
print('Best XGBoost CV R2 :', round(xgb_search.best_score_, 4))
print('Best params:', xgb_search.best_params_)

In [ ]:
best_xgb = xgb_search.best_estimator_
res = evaluate_pipeline('XGBoost (RandomizedSearch tuned)', best_xgb)
results.append(res)

## 6. LightGBM — Optuna Bayesian Optimization

In [ ]:
# Pre-transform features once for Optuna (faster)
preprocessor_fit = get_preprocessor()
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)
X_train_tf = preprocessor_fit.fit_transform(X_train_raw, y_train_raw)
X_test_tf  = preprocessor_fit.transform(X_test_raw)

def lgb_objective(trial):
    params = {
        'n_estimators'     : trial.suggest_int('n_estimators', 100, 1000),
        'max_depth'        : trial.suggest_int('max_depth', 3, 12),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'num_leaves'       : trial.suggest_int('num_leaves', 20, 300),
        'subsample'        : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state'     : 42,
        'n_jobs'           : -1,
        'verbose'          : -1,
    }
    model = lgb.LGBMRegressor(**params)
    scores = cross_val_score(model, X_train_tf, y_train_raw, cv=5, scoring='r2', n_jobs=-1)
    return scores.mean()

study_lgb = optuna.create_study(direction='maximize')
study_lgb.optimize(lgb_objective, n_trials=60, show_progress_bar=False)

print('Best LightGBM CV R2 :', round(study_lgb.best_value, 4))
print('Best params:', study_lgb.best_params)

In [ ]:
best_lgb_params = study_lgb.best_params
best_lgb_params['verbose'] = -1
best_lgb_model = lgb.LGBMRegressor(**best_lgb_params)

lgb_pipe = Pipeline([
    ('pre', get_preprocessor()),
    ('reg', best_lgb_model)
])
res = evaluate_pipeline('LightGBM (Optuna tuned)', lgb_pipe)
results.append(res)

## 7. CatBoost — Optuna Bayesian Optimization

In [ ]:
def catboost_objective(trial):
    params = {
        'iterations'   : trial.suggest_int('iterations', 100, 800),
        'depth'        : trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg'  : trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'subsample'    : trial.suggest_float('subsample', 0.5, 1.0),
        'random_state' : 42,
        'verbose'      : 0,
    }
    model = CatBoostRegressor(**params)
    scores = cross_val_score(model, X_train_tf, y_train_raw, cv=5, scoring='r2', n_jobs=-1)
    return scores.mean()

study_cat = optuna.create_study(direction='maximize')
study_cat.optimize(catboost_objective, n_trials=40, show_progress_bar=False)

print('Best CatBoost CV R2 :', round(study_cat.best_value, 4))
print('Best params:', study_cat.best_params)

In [ ]:
best_cat_params = study_cat.best_params
best_cat_params['verbose'] = 0
best_cat_params['random_state'] = 42
best_cat_model = CatBoostRegressor(**best_cat_params)

cat_pipe = Pipeline([
    ('pre', get_preprocessor()),
    ('reg', best_cat_model)
])
res = evaluate_pipeline('CatBoost (Optuna tuned)', cat_pipe)
results.append(res)

## 8. Stacking Ensemble

Combine tuned XGBoost + LightGBM + Extra Trees as base learners, with Ridge as meta-learner. Stacking often reduces MAE by 5-15% over the best single model.

In [ ]:
# Use the best hyperparams found above
xgb_best_p = xgb_search.best_params_

# Strip pipeline prefix from keys
xgb_model_params = {k.replace('reg__', ''): v for k, v in xgb_best_p.items()}
xgb_final = XGBRegressor(**xgb_model_params, random_state=42, n_jobs=-1, verbosity=0)

lgb_final = lgb.LGBMRegressor(**{k: v for k, v in study_lgb.best_params.items()},
                               verbose=-1, random_state=42, n_jobs=-1)
et_final  = ExtraTreesRegressor(n_estimators=300, random_state=42, n_jobs=-1)
cat_final = CatBoostRegressor(**{k: v for k, v in study_cat.best_params.items()},
                               verbose=0, random_state=42)

stacker = StackingRegressor(
    estimators=[
        ('xgb', xgb_final),
        ('lgb', lgb_final),
        ('et',  et_final),
        ('cat', cat_final),
    ],
    final_estimator=Ridge(alpha=1.0),
    cv=5,
    n_jobs=-1
)

stack_pipe = Pipeline([
    ('pre', get_preprocessor()),
    ('reg', stacker)
])

print('Fitting stacking ensemble (this takes a few minutes)...')
res = evaluate_pipeline('Stacking (XGB+LGB+ET+CB → Ridge)', stack_pipe)
results.append(res)

## 9. Results Summary

In [ ]:
results_df = pd.DataFrame([
    {'Model': r['name'], 'MAE': r['mae'], 'RMSE': r['rmse'], 'R2': r['r2'], 'MAPE%': r['mape'], 'CV_R2': r['cv_r2']}
    for r in results
]).sort_values('MAE').reset_index(drop=True)

print('\n=== RESULTS SUMMARY (sorted by MAE) ===')
print(results_df.to_string(index=False))
print(f'\nPrevious best MAE: 0.4718 (Extra Trees, OHE strategy)')
print(f'New best MAE     : {results_df["MAE"].min():.4f} ({results_df.iloc[0]["Model"]})')
print(f'Improvement      : {(0.4718 - results_df["MAE"].min()) / 0.4718 * 100:.1f}%')

## 10. Error Analysis on Best Model

In [ ]:
import matplotlib.pyplot as plt

best_result = min(results, key=lambda r: r['mae'])
best_pipe   = best_result['pipeline']

X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42)
best_pipe.fit(X_train, y_train)

y_pred_log  = best_pipe.predict(X_test)
y_pred_orig = np.expm1(y_pred_log)
y_true_orig = np.expm1(y_test)

errors = np.abs(y_true_orig.values - y_pred_orig)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(y_true_orig, y_pred_orig, alpha=0.4, s=15, color='steelblue')
lim = max(y_true_orig.max(), y_pred_orig.max())
axes[0].plot([0, lim], [0, lim], 'r--', lw=1.5)
axes[0].set_xlabel('Actual Price (Cr)')
axes[0].set_ylabel('Predicted Price (Cr)')
axes[0].set_title(f'Actual vs Predicted\n{best_result["name"]}')

residuals = y_true_orig.values - y_pred_orig
axes[1].scatter(y_pred_orig, residuals, alpha=0.4, s=15, color='steelblue')
axes[1].axhline(0, color='r', linestyle='--')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residuals vs Predicted')

axes[2].hist(errors, bins=40, color='steelblue', edgecolor='white')
axes[2].axvline(errors.mean(), color='r', linestyle='--', label=f'Mean={errors.mean():.3f}')
axes[2].axvline(np.median(errors), color='orange', linestyle='--', label=f'Median={np.median(errors):.3f}')
axes[2].set_xlabel('Absolute Error (Cr)')
axes[2].set_ylabel('Count')
axes[2].set_title('Error Distribution')
axes[2].legend()

plt.tight_layout()
plt.savefig('error_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'\nError percentiles:')
for p in [50, 75, 90, 95]:
    print(f'  P{p}: {np.percentile(errors, p):.3f} Cr')

## 11. Export Best Model

In [ ]:
# Fit best pipeline on full dataset before export
best_pipe.fit(X, y_log)

with open('pipeline_advanced.pkl', 'wb') as f:
    pickle.dump(best_pipe, f)
with open('df_advanced.pkl', 'wb') as f:
    pickle.dump(X, f)

print(f'Exported: pipeline_advanced.pkl')
print(f'Best model: {best_result["name"]}')
print(f'Test MAE   : {best_result["mae"]:.4f} Crores')
print(f'Test RMSE  : {best_result["rmse"]:.4f} Crores')
print(f'Test R²    : {best_result["r2"]:.4f}')

## 12. Sanity Check — Sample Prediction

In [ ]:
def predict_price(property_type, sector, bedRoom, bathroom, balcony,
                  agePossession, built_up_area, servant_room=0, store_room=0,
                  furnishing='unfurnished', luxury='Low', floor_cat='Mid Floor'):
    sample = pd.DataFrame([{
        'property_type' : property_type,
        'sector'        : sector,
        'bedRoom'       : bedRoom,
        'bathroom'      : bathroom,
        'balcony'       : str(balcony),
        'agePossession' : agePossession,
        'built_up_area' : built_up_area,
        'servant room'  : servant_room,
        'store room'    : store_room,
        'furnishing_type': furnishing,
        'luxury_category': luxury,
        'floor_category' : floor_cat,
    }])
    # Add engineered features
    sample = engineer_features(sample)
    return np.expm1(best_pipe.predict(sample))[0]

examples = [
    ('house',  'sector 102', 4, 3, '3+', 'New Property',    2750, 0, 0, 'unfurnished', 'Low',    'Low Floor'),
    ('flat',   'sector 36',  3, 2, '2',  'New Property',     850, 0, 0, 'unfurnished', 'Low',    'Low Floor'),
    ('flat',   'sector 47',  4, 4, '3+', 'Relatively New', 2200, 1, 1, 'semifurnished','High',  'Mid Floor'),
]

print('Sample predictions:')
for args in examples:
    pred = predict_price(*args)
    print(f'  {args[0]:5s} {args[1]:15s} {args[2]}BHK {args[6]}sqft → Rs {pred:.2f} Cr')